# E001 · 설정과 결과 분석

[실험 안내](README.md) · [설정](config.json) · [검증 기록](validation.md)

이 노트북은 설정과 이미 저장된 결과만 읽습니다. 데이터 다운로드, 모델 로딩, 학습, 서비스 DB 변경은 하지 않습니다. 첫 코드 셀의 `CONFIG_PATH`로 데이터 영역을 선택하세요.

Kernel은 지정한 `.venv/Scripts/python.exe`를 선택하고 위에서부터 실행합니다.


In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
root = next((p for p in (current, *current.parents)
             if (p / 'experiments/e001_e5/config.json').is_file()), None)
if root is None:
    raise RuntimeError('프로젝트 루트 또는 e001_e5 폴더에서 노트북을 여세요.')
CONFIG_PATH = root / 'experiments/e001_e5/config.json'
# Smoke 결과: root / 'data/experiments/e001-smoke/<UTC시간>/config.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
runs_dir = root / config['paths']['runs']
print('설정:', CONFIG_PATH)
print('결과 폴더:', runs_dir)


## 1. 이번 실험의 설정

`ready`는 실행 코드와 설정이 준비됐다는 뜻입니다. learning rate와 batch size는 초기 값이며 성능이나 최대 길이 GPU 메모리에 맞게 조정할 수 있습니다. 모델 revision, prefix, 데이터 분할, 실제 배치, 평가 단위를 확인하세요.


In [ ]:
print(json.dumps(config, ensure_ascii=False, indent=2))


## 2. 데이터와 결과가 준비됐는지 확인

`raw`는 원본, `prepared`는 고정된 전처리 결과, `runs`는 실행별 결과입니다. 폴더가 있어도 내용이 없으면 데이터가 준비된 것은 아닙니다.


In [ ]:
for name, relative in config['paths'].items():
    folder = root / relative
    count = sum(1 for _ in folder.iterdir()) if folder.is_dir() else 0
    print(f'{name}: {relative} | 존재={folder.is_dir()} | 바로 아래 항목={count}')
prepared_manifest = root / config['paths']['prepared'] / 'manifest.json'
if prepared_manifest.is_file():
    data = json.loads(prepared_manifest.read_text(encoding='utf-8'))
    print('데이터:', data['status'], data.get('dataset_id'))
    print('포함/제외:', data.get('counts'))
    print('부분 평가:', data.get('evaluation_subset'))
    print('학습/검증 필터:', data.get('filter_stats'))


## 3. 실행별 최종 검색 성능 비교

Recall@5는 찾은 정답 비율, MRR@10은 첫 정답 순위, nDCG@10은 관련 결과의 순서를 봅니다. 모두 높을수록 좋습니다. 같은 데이터·후보군·평가 단위끼리 비교하세요.

학습 loss와 다른 지표입니다. Baseline에는 학습 history가 없어도 정상입니다.


In [ ]:
run_dirs = sorted(p for p in runs_dir.iterdir() if p.is_dir()) if runs_dir.is_dir() else []
run_states = []
groups = {}
for folder in run_dirs:
    manifest_path = folder / 'manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else {}
    run_states.append({'run': folder.name, 'kind': manifest.get('kind'),
                       'status': manifest.get('status'), 'device': manifest.get('device')})
    metrics_path = folder / 'metrics.json'
    if metrics_path.is_file():
        result = json.loads(metrics_path.read_text(encoding='utf-8'))
        key = (result['dataset_id'], result['split'], result['evaluation_unit'])
        groups.setdefault(key, []).append({'run': folder.name, 'query_count': result['query_count'],
            'subset': result.get('evaluation_subset', False),
            'excluded_queries': result.get('excluded_queries', 0), **result['metrics']})
if run_states:
    display(pd.DataFrame(run_states).set_index('run'))
for (dataset_id, split, unit), rows in groups.items():
    print('동일 비교군:', dataset_id, split, unit)
    display(pd.DataFrame(rows).set_index('run'))
if not groups:
    print('아직 평가 결과가 없습니다. prepare → baseline 평가 → train → 학습 모델 평가 순서로 실행하세요.')


## 4. 분석할 실행 선택

`RUN_NAME`은 학습 실행, `EVAL_RUN_NAME`은 검색 평가 실행입니다. Smoke는 각각 `train`, `tuned`를 넣습니다. 실제 사용한 설정과 상태는 각 run 내부 파일에서 읽습니다.


In [ ]:
RUN_NAME = 'ft_001'
EVAL_RUN_NAME = 'tuned_001'
selected_run = runs_dir / RUN_NAME
selected_eval = runs_dir / EVAL_RUN_NAME
for folder in (selected_run, selected_eval):
    print('선택:', folder.name)
    for name in ('config.json', 'manifest.json', 'best.json'):
        saved = folder / name
        if saved.is_file():
            print(name, json.dumps(json.loads(saved.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
    if not folder.is_dir():
        print('아직 해당 실행이 없습니다. 위 표의 run 이름을 넣으세요.')


## 5. 학습 loss와 validation 검색 지표

Train InfoNCE는 배치 negative, valid InfoNCE는 valid 전체 후보 negative를 사용합니다. **두 loss의 크기는 직접 비교하지 않고** 각각의 변화를 봅니다. 최종 모델은 valid MRR@10으로 선택합니다.

`history.csv`는 step마다 loss·learning rate·validation 지표를 저장하며, 검증하지 않은 step의 검증 값은 빈칸입니다.


In [ ]:
history_path = selected_run / 'history.csv'
if history_path.is_file():
    import matplotlib.pyplot as plt
    history = pd.read_csv(history_path)
    display(history.tail(10))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['step'], history['train_loss'])
    axes[0].set(title='Train InfoNCE (batch negatives)', xlabel='Step', ylabel='Loss')
    valid = history.dropna(subset=['val_loss'])
    axes[1].plot(valid['step'], valid['val_loss'], marker='o')
    axes[1].set(title='Valid InfoNCE (all valid negatives)', xlabel='Step', ylabel='Loss')
    valid = history.dropna(subset=['validation_mrr_at_10'])
    axes[2].plot(valid['step'], valid['validation_mrr_at_10'], marker='o')
    axes[2].set(title='Validation retrieval', xlabel='Step', ylabel='MRR@10', ylim=(0, 1.05))
    plt.tight_layout()
    plt.show()
else:
    print('해당 실행의 학습 history가 없습니다. 학습 run을 선택하세요.')


## 6. 질문별 검색 결과와 정답 비교

평가 run의 질문·정답 파일·순위·파일 점수·대표 청크 라인을 확인합니다. LCA의 정답 단위는 파일이며, 대표 청크가 정답 문단이라는 의미는 아닙니다.

`QUERY_ID=None`이면 처음 3개 질문, 실제 ID를 넣으면 해당 질문을 표시합니다.


In [ ]:
QUERY_ID = None
rankings_path = selected_eval / 'rankings.jsonl'
if rankings_path.is_file():
    shown = 0
    with rankings_path.open(encoding='utf-8') as stream:
        for line in stream:
            if not line.strip():
                continue
            item = json.loads(line)
            if QUERY_ID is not None and item['query_id'] != QUERY_ID:
                continue
            print('Query:', item['query_id'], item['question'])
            print('Repository/commit:', item.get('repo'), item.get('commit_sha'))
            print('정답:', item['relevant_ids'])
            table = pd.DataFrame(item['retrieved'])
            columns = [c for c in ['rank', 'path', 'score', 'relevant', 'chunk_id', 'start_line', 'end_line'] if c in table]
            display(table[columns])
            for result in item['retrieved'][:3]:
                if 'text' in result:
                    print(result['path'], f"lines {result['start_line']}–{result['end_line']}")
                    print(result['text'][:1500])
            shown += 1
            if QUERY_ID is not None or shown >= 3:
                break
    if shown == 0:
        print('선택한 질문의 검색 결과가 없습니다.')
else:
    print('아직 질문별 검색 순위가 없습니다. 평가 run을 선택하세요.')


## 해석과 다음 실험

동일한 `dataset_id`·split·평가 단위의 baseline과 tuned를 비교합니다. 성능이 나빠진 질문도 직접 읽고 학습률·배치·temperature를 하나씩 조정합니다. 최종 test를 반복해서 보고 설정을 고르면 test에도 적응할 수 있으므로 개선 확인용 독립 질문을 추가하세요.

Smoke는 코드 연결 검증용입니다. 서비스 적용 판단에는 정식 실험과 한국어/영어 서비스 정답 질문이 필요합니다. 적용 시 새 모델로 서비스 문서 전체를 다시 벡터화합니다.
